# 02 — Latent Class Model & Functional Class Validation

This notebook fits a **2-class Negative Binomial latent class model** and tests
whether the discovered classes align with the known **Functional Class (FC)** groups.

**Hypothesis:** road segments belong to latent risk sub-populations that are at least
partially explained by their functional classification (freeway, arterial, collector, etc.).

**What you will do:**
1. Constrain `FC_ENCODED` to drive class membership probability only (not the outcome)
2. Run a 2-class LC structure search
3. Extract posterior class membership probabilities for each site
4. Assign each site to its most-likely class
5. Cross-tabulate predicted class vs actual FC label (chi-square test)
6. Fit the pre-specified reference model from the literature

**Previous notebook:** [01_crash_frequency_search.ipynb](01_crash_frequency_search.ipynb)
**Next notebook:** [03_cmf_aadt_search.ipynb](03_cmf_aadt_search.ipynb)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example16_3_model_data,
    load_book_latent_class_spec,
    describe_book_latent_class_spec,
    get_help,
)

# Uncomment to read the full latent class guide:
# get_help('latent_class')

print('Imports OK')

## 1 — Load data and inspect functional class distribution

In [ ]:
df = load_example16_3_model_data()

print(f'Dataset: {len(df)} road segments, {df.shape[1]} columns')
print()
print('Functional class distribution (FC_LABEL):')
fc_dist = df['FC_LABEL'].value_counts().sort_index()
print(fc_dist.to_string())
print()
print('FC_ENCODED mapping (0-indexed label for modelling):')
print(df[['FC', 'FC_ENCODED', 'FC_LABEL']].drop_duplicates().sort_values('FC_ENCODED'))
print()
print('Outcome (FREQ) summary:')
print(df['FREQ'].describe().round(2))

In [ ]:
df[['ID', 'FREQ', 'FC_LABEL', 'FC_ENCODED', 'AADT', 'OFFSET']].head(8)

## 2 — Latent class constraints

**Key design decision:** `FC_ENCODED` enters **only** the class-membership logit equation
(role 7).  It does **not** directly affect the crash-count outcome — that would make
the test circular.  Instead, FC shapes the probability that a segment belongs to
the high-risk or low-risk latent class.

If the latent classes recover the FC grouping without FC in the outcome equation,
it means the structural random-parameter heterogeneity is genuinely associated with
functional class — a meaningful finding.

In [ ]:
variables = [
    'AADT',       # primary traffic volume
    'LENGTH',     # segment length
    'SPEED',      # posted speed
    'CURVES',     # horizontal curvature
    'URB',        # urban indicator
    'ACCESS',     # access control
    'GRADEBR',    # grade break
    'SLOPE',      # slope
    'AVEPRE',     # annual precipitation
    'FC_ENCODED', # ← membership only (not in outcome equation)
]

c = (
    ModelConstraints()

    # FC_ENCODED may only drive class membership probability
    # It cannot be a fixed predictor or random effect in the outcome
    .membership_only('FC_ENCODED')

    # Exposure must always be included
    .force_include('OFFSET')

    # Road geometry: not plausible as ZI terms
    .no_zi('LENGTH', 'CURVES', 'GRADEBR', 'WIDTH', 'SLOPE')

    # Binary dummies: no individual taste variation
    .no_random('URB', 'GRADEBR')

    # Allow random curvature (lognormal = must be positive)
    .allow_random('CURVES', distributions=['lognormal'])
)

print('Active constraints:')
print(c)

## 3 — Build the LC evaluator and run the search

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='ID',
    y_col='FREQ',
    offset_col='OFFSET',
)

# ── Key settings for LC search ────────────────────────────────────────────────
#
#   max_latent_classes=2   : allow up to 2 latent classes in the decision space
#   default_roles includes 7 and 8 : search may assign membership roles
#   R=150                  : fewer draws for faster LC search
#                            (LC EM is more expensive than single-class)

evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    default_roles=[0, 1, 2, 3, 5, 7, 8],  # include membership roles!
    max_latent_classes=2,
    mode='single',
    R=150,
)

print('LC evaluator built.')
print('Variables:', variables)
print('Roles allowed:', [0, 1, 2, 3, 5, 7, 8])
print('  0=excluded, 1=fixed, 2=rdm-ind, 3=rdm-cor, 5=hetro, 7=membership, 8=membership+fixed')

In [ ]:
print('Running 2-class LC search (max_iter=30 — demo run) ...')
print('Increase to 200+ for a production search.')
print()

result = builder.run(
    evaluator=evaluator,
    algo='sa',
    max_iter=30,
    seed=1,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name='lc_fc_validation',
        search_description='2-class LC NB search — FC validation on Example 16-3',
    ),
)

print('─' * 60)
print(f'  Algorithm        : {result.algorithm}')
print(f'  Best BIC         : {result.best_score:.3f}')
print(f'  Iterations run   : {result.iteration}')
print(f'  Time elapsed     : {result.elapsed_time:.1f} s')
print('─' * 60)
print()
print('Best structure found:')
for k, v in result.best_spec.items():
    if v:
        print(f'  {k:<20}: {v}')

## 4 — Extract class membership probabilities

`compute_latent_class_probabilities()` returns the posterior probability that
each segment belongs to each class, given its observed covariates and crash count.

In [ ]:
# Re-fit the best structure with more draws for accurate class probabilities
print('Re-fitting with R=300 for class probability extraction ...')

fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='nb',
    R=300,
)

print('Fit complete.')

In [ ]:
# Extract posterior class membership probabilities
class_probs = builder.compute_latent_class_probabilities(
    fit,
    true_class_col='FC_ENCODED',   # adds FC_ENCODED to the output for comparison
)

print(f'Class probability table: {class_probs.shape[0]} rows × {class_probs.shape[1]} columns')
print()
print('Columns:', class_probs.columns.tolist())
print()
class_probs.head(8)

In [ ]:
# Assign each site to its most-likely class (1-indexed)
prob_cols = [c for c in class_probs.columns if c.startswith('class_') and c.endswith('_prob')]
class_probs['predicted_class'] = class_probs[prob_cols].to_numpy().argmax(axis=1)  # 0-indexed
class_probs['max_prob'] = class_probs[prob_cols].to_numpy().max(axis=1)            # confidence

print('Predicted class distribution:')
print(class_probs['predicted_class'].value_counts().sort_index())
print()
print(f'Mean assignment confidence: {class_probs["max_prob"].mean():.3f}')
print(f'Sites with >90% confidence: {(class_probs["max_prob"] > 0.9).sum()}')

## 5 — Validate: predicted class vs actual functional class

In [ ]:
# Merge with original FC_LABEL for readable output
merged = class_probs.merge(
    df[['ID', 'FC_LABEL', 'FC_ENCODED']].rename(columns={'FC_ENCODED': 'FC_ENC_orig'}),
    on='ID', how='left'
)

# Cross-tabulation: predicted class vs actual FC label
ct = pd.crosstab(
    merged['FC_LABEL'],
    merged['predicted_class'],
    rownames=['Functional Class'],
    colnames=['Predicted Class'],
)

print('Contingency table (counts):')
print(ct)
print()

print('Row proportions  Pr[predicted class | FC]:')
print(ct.div(ct.sum(axis=1), axis=0).round(3))

In [ ]:
# Chi-square test of independence
chi2, p, dof, expected = chi2_contingency(ct)

print('─' * 60)
print(f'Chi-square statistic : {chi2:.3f}')
print(f'Degrees of freedom   : {dof}')
print(f'p-value              : {p:.4f}')
print('─' * 60)

if p < 0.001:
    print('RESULT: Very strong association between predicted classes and FC (p < 0.001)')
    print('The latent classes are capturing functional-class-driven heterogeneity.')
elif p < 0.05:
    print(f'RESULT: Significant association between predicted classes and FC (p = {p:.4f})')
    print('The latent classes show meaningful alignment with functional class.')
else:
    print(f'RESULT: No significant association (p = {p:.4f})')
    print('The latent classes do not clearly align with functional class.')
    print('Consider: more iterations, different membership variables, or a different K.')

## 6 — Compare 1-class vs 2-class BIC

Run the same structure as a 1-class model to check whether the extra class is justified.

In [ ]:
# Build a 1-class version of the same structure
one_class_spec = dict(result.best_spec)   # copy
one_class_spec['latent_classes'] = 1
one_class_spec['membership_terms'] = []

fit_1c = builder.fit_manual_model(
    manual_spec=one_class_spec,
    model='nb',
    R=300,
)

# Extract BIC values from fit results
# (the exact key depends on the fit output format)
print('─' * 60)
print('BIC comparison:')
print(f'  1-class model BIC : (see fit_1c output below)')
print(f'  2-class model BIC : {result.best_score:.3f}')
print('─' * 60)
print()
print('1-class fit:')
print(fit_1c)
print()
print('Rule of thumb: accept 2-class model if ΔBIC > 10  (lower BIC = better)')

## 7 — Fit the pre-specified reference model

The package includes a reference 2-class NB specification from the crash-frequency
literature.  Use `describe_book_latent_class_spec()` to see what it contains,
then fit it to compare against your searched structure.

In [ ]:
# Describe the reference specification
describe_book_latent_class_spec()

In [ ]:
# Load and fit the reference spec
ref_spec = load_book_latent_class_spec()

print('Reference spec:')
for k, v in ref_spec.items():
    print(f'  {k:<20}: {v}')
print()

ref_fit = builder.fit_manual_model(
    manual_spec=ref_spec,
    model='nb',
    R=300,
)

print('─' * 60)
print('Reference model fit:')
print('─' * 60)
print(ref_fit)

## Interpretation guide

| Finding | Interpretation |
| --- | --- |
| p < 0.05 in chi-square | Latent classes are significantly associated with FC — classes capture functional heterogeneity |
| One FC category dominates a class | That functional class has a distinct crash-risk profile |
| ΔBIC > 10 for 2-class vs 1-class | Adding the second class improves fit sufficiently to justify the extra parameters |
| Class 1 >> Class 2 in size | Check whether the minority class is supported — compare BIC and inspect confidence scores |
| High mean confidence (>0.8) | Class assignments are well-determined — the classes are well-separated |

**Next:** [03_cmf_aadt_search.ipynb](03_cmf_aadt_search.ipynb) — fit a CMF model
with AADT as the primary scaling term.